In [1]:
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import make_column_transformer
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix
from imblearn.under_sampling import RandomUnderSampler
import pandas as pd
import numpy as np

from dataprocessing import data_noutliers

c:\Users\yzhen\OneDrive\Documents\MSPPM-DA\SP2025\ML\project - education\dataprocessing.py:52: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data['school_state_region'] = data['school_state'].map(state_to_region)
c:\Users\yzhen\OneDrive\Documents\MSPPM-DA\SP2025\ML\project - education\dataprocessing.py:57: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data['fully_funded'] = data['fully_funded'].map(binary_map)
c:\Users\yzhen\OneDrive\Documents\MSPPM-DA\SP2025\ML\project - education\dataprocessing.py:61: Set

In [2]:
data_noutliers.shape

(485604, 28)

**School Metro & Poverty Levels**

Baseline

In [3]:
features = ['school_metro','poverty_level']

In [4]:
#one hot encoding

ohe = OneHotEncoder(sparse_output=False)
ct = make_column_transformer(
       (ohe, features),
       remainder = 'passthrough'
)

features_encoded = ct.fit_transform(data_noutliers[features])

#training

X_train, X_test, y_train, y_test = train_test_split(features_encoded, data_noutliers['fully_funded'], test_size=0.2, random_state=42,stratify=data_noutliers['fully_funded'])

clf = HistGradientBoostingClassifier().fit(X_train,y_train)

#testing

y_pred = clf.predict(X_test)

print(classification_report(y_test, y_pred))

cm = confusion_matrix(y_test, y_pred)
TN, FP, FN, TP = cm.ravel()

specificity = TN / (TN + FP)
print(f"Specificity: {specificity:.2f}")

auc_score = roc_auc_score(y_test, y_pred)
print(f"ROC-AUC: {auc_score}\n")

              precision    recall  f1-score   support

           0       0.00      0.00      0.00     27674
           1       0.72      1.00      0.83     69447

    accuracy                           0.72     97121
   macro avg       0.36      0.50      0.42     97121
weighted avg       0.51      0.72      0.60     97121

Specificity: 0.00
ROC-AUC: 0.5



c:\Users\yzhen\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\yzhen\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\yzhen\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


Random Under Sampler for Class Unbalances

In [5]:
#training

rus = RandomUnderSampler(random_state=42)
X_train_rus, y_train_rus = rus.fit_resample(X_train, y_train)

#testing

clf_rus = HistGradientBoostingClassifier().fit(X_train_rus,y_train_rus)
y_pred_rus = clf_rus.predict(X_test)

print(classification_report(y_test, y_pred_rus))

cm_rus = confusion_matrix(y_test, y_pred_rus)
TN, FP, FN, TP = cm_rus.ravel()

specificity = TN / (TN + FP)
print(f"Specificity: {specificity:.2f}")

auc_score = roc_auc_score(y_test, y_pred)
print(f"ROC-AUC: {auc_score}\n")

c:\Users\yzhen\anaconda3\Lib\site-packages\sklearn\utils\deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
c:\Users\yzhen\anaconda3\Lib\site-packages\sklearn\base.py:484: FutureWarning: `BaseEstimator._check_n_features` is deprecated in 1.6 and will be removed in 1.7. Use `sklearn.utils.validation._check_n_features` instead.
  warnings.warn(
c:\Users\yzhen\anaconda3\Lib\site-packages\sklearn\base.py:493: FutureWarning: `BaseEstimator._check_feature_names` is deprecated in 1.6 and will be removed in 1.7. Use `sklearn.utils.validation._check_feature_names` instead.
  warnings.warn(


              precision    recall  f1-score   support

           0       0.33      0.53      0.41     27674
           1       0.75      0.58      0.65     69447

    accuracy                           0.56     97121
   macro avg       0.54      0.55      0.53     97121
weighted avg       0.63      0.56      0.58     97121

Specificity: 0.53
ROC-AUC: 0.5



In [6]:
#checking distribution

print(f"\nBefore Undersampling - Training examples: {len(X_train)}")
print(f"Class distribution: {np.bincount(y_train)}")

print(f"\nAfter Undersampling - Training examples: {len(X_train_rus)}")
print(f"Class distribution: {np.bincount(y_train_rus)}")


Before Undersampling - Training examples: 388483
Class distribution: [110694 277789]

After Undersampling - Training examples: 221388
Class distribution: [110694 110694]


**RUS: Adding Resource Type & Primary Subject**

In [7]:
features = ['school_metro','poverty_level','resource_type','primary_focus_subject']

In [8]:
#one hot encoding

ohe = OneHotEncoder(sparse_output=False)
ct = make_column_transformer(
       (ohe, features),
       remainder = 'passthrough'
)

features_encoded = ct.fit_transform(data_noutliers[features])

#training

X_train, X_test, y_train, y_test = train_test_split(features_encoded, data_noutliers['fully_funded'], test_size=0.2, random_state=42,stratify=data_noutliers['fully_funded'])

#undersampling

rus = RandomUnderSampler(random_state=42)
X_train_rus, y_train_rus = rus.fit_resample(X_train, y_train)

#testing

clf_rus = HistGradientBoostingClassifier().fit(X_train_rus,y_train_rus)
y_pred_rus = clf_rus.predict(X_test)

print(classification_report(y_test, y_pred_rus))

cm_rus = confusion_matrix(y_test, y_pred_rus)
TN, FP, FN, TP = cm_rus.ravel()

specificity = TN / (TN + FP)
print(f"Specificity: {specificity:.2f}")

auc_score = roc_auc_score(y_test, y_pred)
print(f"ROC-AUC: {auc_score}\n")

c:\Users\yzhen\anaconda3\Lib\site-packages\sklearn\utils\deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
c:\Users\yzhen\anaconda3\Lib\site-packages\sklearn\base.py:484: FutureWarning: `BaseEstimator._check_n_features` is deprecated in 1.6 and will be removed in 1.7. Use `sklearn.utils.validation._check_n_features` instead.
  warnings.warn(
c:\Users\yzhen\anaconda3\Lib\site-packages\sklearn\base.py:493: FutureWarning: `BaseEstimator._check_feature_names` is deprecated in 1.6 and will be removed in 1.7. Use `sklearn.utils.validation._check_feature_names` instead.
  warnings.warn(


              precision    recall  f1-score   support

           0       0.36      0.62      0.45     27674
           1       0.78      0.55      0.65     69447

    accuracy                           0.57     97121
   macro avg       0.57      0.59      0.55     97121
weighted avg       0.66      0.57      0.59     97121

Specificity: 0.62
ROC-AUC: 0.5



**RUS: Add Students Reached + Funding Request Amt**

In [9]:
features = ['school_metro','poverty_level','resource_type','primary_focus_subject']

In [19]:
#one hot encoding and adding numerical features

ohe = OneHotEncoder(sparse_output=False)
ct = make_column_transformer(
       (ohe, features),
       remainder = 'passthrough'
)

features_encoded = ct.fit_transform(data_noutliers[features])

#add numerical features
features_encoded = features_encoded.tolist()

for i in range(len(features_encoded)):
    features_encoded[i] = features_encoded[i] + [data_noutliers.loc[i,'students_reached_scaled']] + [data_noutliers.loc[i,'total_price_excluding_optional_support_scaled']]

features_encoded = np.array(features_encoded)

In [20]:
#training 

X_train, X_test, y_train, y_test = train_test_split(features_encoded, data_noutliers['fully_funded'], test_size=0.2, random_state=42,stratify=data_noutliers['fully_funded'])

#undersampling

rus = RandomUnderSampler(random_state=42)
X_train_rus, y_train_rus = rus.fit_resample(X_train, y_train)

#testing

clf_rus = HistGradientBoostingClassifier().fit(X_train_rus,y_train_rus)
y_pred_rus = clf_rus.predict(X_test)

print(classification_report(y_test, y_pred_rus))

cm_rus = confusion_matrix(y_test, y_pred_rus)
TN, FP, FN, TP = cm_rus.ravel()

specificity = TN / (TN + FP)
print(f"Specificity: {specificity:.2f}")

auc_score = roc_auc_score(y_test, y_pred)
print(f"ROC-AUC: {auc_score}\n")

c:\Users\yzhen\anaconda3\Lib\site-packages\sklearn\utils\deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
c:\Users\yzhen\anaconda3\Lib\site-packages\sklearn\base.py:484: FutureWarning: `BaseEstimator._check_n_features` is deprecated in 1.6 and will be removed in 1.7. Use `sklearn.utils.validation._check_n_features` instead.
  warnings.warn(
c:\Users\yzhen\anaconda3\Lib\site-packages\sklearn\base.py:493: FutureWarning: `BaseEstimator._check_feature_names` is deprecated in 1.6 and will be removed in 1.7. Use `sklearn.utils.validation._check_feature_names` instead.
  warnings.warn(


              precision    recall  f1-score   support

           0       0.37      0.72      0.49     27674
           1       0.82      0.51      0.63     69447

    accuracy                           0.57     97121
   macro avg       0.59      0.61      0.56     97121
weighted avg       0.69      0.57      0.59     97121

Specificity: 0.72
ROC-AUC: 0.5



**RUS: Add Matching Statuses, Removing School Poverty Status and School Location Type**

In [21]:
features = ['eligible_double_your_impact_match','eligible_almost_home_match','primary_focus_subject','resource_type']

In [22]:
#one hot encoding

ohe = OneHotEncoder(sparse_output=False)
ct = make_column_transformer(
       (ohe, features),
       remainder = 'passthrough'
)

features_encoded = ct.fit_transform(data_noutliers[features])

#add numerical features
features_encoded = features_encoded.tolist()

for i in range(len(features_encoded)):
    features_encoded[i] = features_encoded[i] + [data_noutliers.loc[i,'students_reached_scaled']] + [data_noutliers.loc[i,'total_price_excluding_optional_support_scaled']]

features_encoded = np.array(features_encoded)

#training 

X_train, X_test, y_train, y_test = train_test_split(features_encoded, data_noutliers['fully_funded'], test_size=0.2, random_state=42,stratify=data_noutliers['fully_funded'])

#undersampling

rus = RandomUnderSampler(random_state=42)
X_train_rus, y_train_rus = rus.fit_resample(X_train, y_train)

#testing

clf_rus = HistGradientBoostingClassifier().fit(X_train_rus,y_train_rus)
y_pred_rus = clf_rus.predict(X_test)

print(classification_report(y_test, y_pred_rus))

cm_rus = confusion_matrix(y_test, y_pred_rus)
TN, FP, FN, TP = cm_rus.ravel()

specificity = TN / (TN + FP)
print(f"Specificity: {specificity:.2f}")

auc_score = roc_auc_score(y_test, y_pred)
print(f"ROC-AUC: {auc_score}\n")

c:\Users\yzhen\anaconda3\Lib\site-packages\sklearn\utils\deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
c:\Users\yzhen\anaconda3\Lib\site-packages\sklearn\base.py:484: FutureWarning: `BaseEstimator._check_n_features` is deprecated in 1.6 and will be removed in 1.7. Use `sklearn.utils.validation._check_n_features` instead.
  warnings.warn(
c:\Users\yzhen\anaconda3\Lib\site-packages\sklearn\base.py:493: FutureWarning: `BaseEstimator._check_feature_names` is deprecated in 1.6 and will be removed in 1.7. Use `sklearn.utils.validation._check_feature_names` instead.
  warnings.warn(


              precision    recall  f1-score   support

           0       0.40      0.70      0.51     27674
           1       0.83      0.57      0.68     69447

    accuracy                           0.61     97121
   macro avg       0.61      0.64      0.59     97121
weighted avg       0.70      0.61      0.63     97121

Specificity: 0.70
ROC-AUC: 0.5



**RUS: School Type**

In [23]:
features = ['school_magnet','school_nlns','school_kipp','school_charter','school_charter_ready_promise']

In [24]:
ohe = OneHotEncoder(sparse_output=False)
ct = make_column_transformer(
       (ohe, features),
       remainder = 'passthrough'
)

features_encoded = ct.fit_transform(data_noutliers[features])

#training 

X_train, X_test, y_train, y_test = train_test_split(features_encoded, data_noutliers['fully_funded'], test_size=0.2, random_state=42,stratify=data_noutliers['fully_funded'])

#undersampling

rus = RandomUnderSampler(random_state=42)
X_train_rus, y_train_rus = rus.fit_resample(X_train, y_train)

#testing

clf_rus = HistGradientBoostingClassifier().fit(X_train_rus,y_train_rus)
y_pred_rus = clf_rus.predict(X_test)

print(classification_report(y_test, y_pred_rus))

cm_rus = confusion_matrix(y_test, y_pred_rus)
TN, FP, FN, TP = cm_rus.ravel()

specificity = TN / (TN + FP)
print(f"Specificity: {specificity:.2f}")

auc_score = roc_auc_score(y_test, y_pred)
print(f"ROC-AUC: {auc_score}\n")

c:\Users\yzhen\anaconda3\Lib\site-packages\sklearn\utils\deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
c:\Users\yzhen\anaconda3\Lib\site-packages\sklearn\base.py:484: FutureWarning: `BaseEstimator._check_n_features` is deprecated in 1.6 and will be removed in 1.7. Use `sklearn.utils.validation._check_n_features` instead.
  warnings.warn(
c:\Users\yzhen\anaconda3\Lib\site-packages\sklearn\base.py:493: FutureWarning: `BaseEstimator._check_feature_names` is deprecated in 1.6 and will be removed in 1.7. Use `sklearn.utils.validation._check_feature_names` instead.
  warnings.warn(


              precision    recall  f1-score   support

           0       0.30      0.83      0.44     27674
           1       0.76      0.21      0.33     69447

    accuracy                           0.39     97121
   macro avg       0.53      0.52      0.38     97121
weighted avg       0.63      0.39      0.36     97121

Specificity: 0.83
ROC-AUC: 0.5

